# Secure Memory and Data Storage

This exercise covers confidential data storage on block device level and the usage of encrypted file containers with a file system inside.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [ ]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "rservice" ssh rservice@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)    

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

## Exercise 5.1: Meet ``dm-crypt``

``cryptsetup`` is the command line tool to interface with ``dm-crypt`` for creating, accessing and managing encrypted block devices, partitions, and file containers. It also supports the Linux Unified Key Setup (LUKS) extension, which stores several meta-data about the encryption setup for ``dm-crypt`` on the disk itself. Further, it abstracts partition and key management in an attempt to improve ease of use.

### Basic Usage of ``cryptsetup``

The command line tool is used as follows:

``cryptsetup <options> <action> <action-specific-options>``

It has compiled-in defaults for the options and the encryption mode, which will be used if no others are specified on the command line. Have a look at ``cryptsetup --help `` to list options, actions and - at the end of the output - its default parameters.

In [ ]:
execute_stm32_command('/usr/sbin/cryptsetup --help')

#### Creation of an Encrypted Container

Open a terminal and connect to your STM32MP1 board by ``ssh root@192.168.13.13``.

In [ ]:
execute_vm_command('gnome-terminal')

A typical use case for ``cryptsetup`` is the creation of an encrypted file container.

Execute all the following commands within the newly created terminal session.

First, generate a random 32MB file ``container.enc`` in the home directory of your STM32MP1 board with ``dd``:

``dd if=/dev/urandom of=container.enc bs=1M count=32``

Generate a random file ``keyfile.key`` containing 256 bits that can serve as a secret key file:

``dd if=/dev/urandom of=keyfile.key bs=32 count=1``

Use ``luksFormat`` to initialize the file container for LUKS in version 2 and your key file:

``cryptsetup -q -v luksFormat --type luks2 container.enc keyfile.key``

Open your LUKS file container ``container.enc`` and map it to the virtual device ``dm_enc``:

``cryptsetup -v -d keyfile.key luksOpen container.enc dm_enc``

#### File System Creation and Mounting

The decrypted block device can be accessed by ``/dev/mapper/dm_enc``.

The following command creates an ``ext2`` file system within the created container:

``mke2fs /dev/mapper/dm_enc``

Subsequently, the created file system can be mounted to a given mount point, e.g. ``/mnt/enc_con``:

``mkdir /mnt/enc_con``

``mount /dev/mapper/dm_enc /mnt/enc_con``

#### Container Analysis

The following code block checks the size of your file container:

``ls -lah container.enc``

The tool ``fdisk`` can be used to show the size of the block device:

``fdisk -l /dev/mapper/dm_enc``

Also, it is interesting to show the available space in the file system within container:

``df -hT /dev/mapper/dm_enc``

#### Clean-up

Finally, unmount the file system and remove the created mount point:

``umount /mnt/enc_con``

``rm -r /mnt/enc_con``

Close the LUKS container, delete it and remove the created key file:

``cryptsetup luksClose dm_enc``

``rm container.enc keyfile.key``

### Analysis of Results

Analyze the sizes of the file container, the virtual block device and the available space within the file system and think about reasons for that!

**Size of the File Container:**

*TODO*

**Size of the Virtual Block Device:**

*TODO*

**Available Space in the File System within the Encrypted Container:**

*TODO*

**Reasons for the Different Sizes**

*TODO*

### After Your Analysis

A wise guy tells you that this result is to be expected with ``luks2`` metadata headers... That's annoying, but it motivates you to do better!

**Tasks:**<br>
- Research why the metadata type ``luks1`` might be an option to solve your problem.
- Create the container again with the metadata type ``luks1`` and analyze the result.

**Size of the NEW Virtual Block Device:**

*TODO*

**Available Space in the NEW File System within the Encrypted Container:**

*TODO*

---

## Exercise 5.2: Custom Options for Container Encryption

The previous example used the default parameters of ``cryptsetup``.

Let's create it again:

``dd if=/dev/urandom of=container.enc bs=1M count=32``

``dd if=/dev/urandom of=keyfile.key bs=32 count=1``

``cryptsetup -q -v luksFormat container.enc keyfile.key``

``cryptsetup -v -d keyfile.key luksOpen container.enc dm_enc``

Then, use the ``cryptsetup status`` command to display parameters like the utilized ciphers:

``cryptsetup status /dev/mapper/dm_enc``

Afterward, close the LUKS container, delete it and remove the created key file:

``cryptsetup luksClose dm_enc``
``rm container.enc keyfile.key``

**Used Block Cipher:**

*TODO*

**Used Operation Mode:**

*TODO*

**Used Key Size:**

*TODO*

### Benchmarking Available Crypto Algorithms

``cryptsetup`` can use various crypto algorithms available from the Linux kernel's crypto API.

Execute the ``cryptsetup benchmark`` command to list the available crypto on your device and to measure its performance:

``cryptsetup benchmark``

### Analysis of Results

The previous output can give guidance on deciding for algorithms and parameters prior to creating encrypted volumes.

1. Which algorithm configuration shows the highest performance on your device?

2. Why does ``cryptsetup`` also test the performance of PBKDF2 and Argon2?

**Best Performing Algorithm on the Given Platform:**

*TODO*

**Reasons for PBKDF2 and Argon2 Benchmarking:**

*TODO*